# 01 - Preprocessing
Carrega o NUAA pelo protocolo oficial, roda MTCNN (margin=40),
salva em data/processed/{train,validation,test}/{live,spoof}/

In [ ]:
# Clona o repositório e instala as dependências.
# facenet-pytorch precisa de --no-deps porque o Colab não tem wheels
# pré-compiladas pras versões antigas de numpy/Pillow que ele pede.

!git clone https://github.com/laianemuckler/liveness-detection.git
%cd liveness-detection

!pip install -r requirements.txt --quiet
!pip install facenet-pytorch==2.6.0 --no-deps --quiet

In [ ]:
# Monta o Drive

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Importa as funções do projeto

import os
from tqdm import tqdm

from src.config import (
    TRAIN_DIR, VAL_DIR, TEST_DIR,
    LABEL_LIVE, LABEL_SPOOF,
)
from src.dataset import (
    load_protocol,
    split_train_validation,
    build_mtcnn,
    align_face,
)

In [ ]:
# Carrega o protocolo e monta o split

protocol = load_protocol()
train_entries, validation_entries = split_train_validation(protocol['train'])
test_entries = protocol['test']

print(f"Train:      {len(train_entries)} images")
print(f"Validation: {len(validation_entries)} images")
print(f"Test:       {len(test_entries)} images")

In [ ]:
# 1. Cria o detector MTCNN (com margin=40, já configurado)
# 2. Para cada imagem original (Client/Imposter):
#    - roda o MTCNN -> detecta e recorta o rosto (imagem NOVA, não a original)
#    - se não detectar rosto, pula e conta como "não detectado"
# 3. Salva essa imagem nova em data/processed/<split>/<live ou spoof>/
#    onde <split> é train, validation ou test

mtcnn = build_mtcnn()

LABEL_TO_FOLDER = {LABEL_LIVE: 'live', LABEL_SPOOF: 'spoof'}


def process_split(entries, output_dir, split_name):
    for label in (LABEL_LIVE, LABEL_SPOOF):
        os.makedirs(os.path.join(output_dir, LABEL_TO_FOLDER[label]), exist_ok=True)

    not_detected = 0
    for path, label, subject_id in tqdm(entries, desc=split_name):
        face_tensor = align_face(mtcnn, path)
        if face_tensor is None:
            not_detected += 1
            continue

        original_filename = os.path.basename(path)
        out_name = f"{subject_id}_{original_filename}"
        out_path = os.path.join(output_dir, LABEL_TO_FOLDER[label], out_name)

        from PIL import Image
        import numpy as np

        img_array = face_tensor.permute(1, 2, 0).numpy()
        img_array = (img_array - img_array.min()) / (img_array.max() - img_array.min())
        img_array = (img_array * 255).astype(np.uint8)
        Image.fromarray(img_array).save(out_path)

    print(f"{split_name}: {len(entries) - not_detected} saved | {not_detected} faces not detected")

In [ ]:
# 1. Chama process_split() três vezes, uma para cada conjunto
# 2. Cada chamada processa TODAS as imagens daquele conjunto (MTCNN + salvar)
# 3. Demora bastante -- está processando as ~12 mil imagens do NUAA

process_split(train_entries, TRAIN_DIR, 'train')
process_split(validation_entries, VAL_DIR, 'validation')
process_split(test_entries, TEST_DIR, 'test')

In [ ]:
# Conferência: conta quantas imagens ficaram salvas em cada pasta

for split_dir, split_name in [(TRAIN_DIR, 'train'), (VAL_DIR, 'validation'), (TEST_DIR, 'test')]:
    n_live = len(os.listdir(os.path.join(split_dir, 'live')))
    n_spoof = len(os.listdir(os.path.join(split_dir, 'spoof')))
    print(f"{split_name}: live={n_live} | spoof={n_spoof} | total={n_live + n_spoof}")